# 🌊 Domácí úkol (Session 2): Neuronové sítě – Klasifikace signálů sonaru (Skála vs. Kovová mina)
**Dataset:** `sonar.csv` (208 měření, 60 frekvenčních pásem sonaru)  
**Cíl úlohy:** Konstrukce a trénování plně propojené dopředné neuronové sítě (Multilayer Perceptron – MLP) v knihovně **Keras / TensorFlow** pro binární klasifikaci objektu na dně moře: odraz od **přírodní skály (`Rock - R`)** vs. **kovového válce / miny (`Mine - M`)**.

Zdroj dat: [UCI Machine Learning Repository – Connectionist Bench (Sonar, Mines vs. Rocks)](https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks).

---
### 📋 Kroky zadání:
1. **Import knihoven:** Importujte třídy z knihovny TensorFlow / Keras (`Sequential`, `Dense`, `Dropout`, `Input`), Pandas, NumPy a Scikit-learn.
2. **Načtení dat:** Načtěte soubor `sonar.csv` pomocí knihovny Pandas (pozor na absenci hlavičky v CSV).
3. **Transformace proměnných:** 
   - Nezávislé proměnné (60 spektrálních pásem): normalizace / standardizace (`StandardScaler`).
   - Závislá proměnná (cíl): převod kategorického textu (`'R'` a `'M'`) na binární číselný indikátor (0 a 1).
4. **Rozdělení dat:** Rozdělte data na trénovací a testovací množinu (`train_test_split`).
5. **Stavba neuronové sítě:** Pomocí tříd z knihovny Keras sestavte vrstvy neuronové sítě s aktivačními funkcemi ReLU a výstupním Sigmoidem.
6. **Kompilace modelu:** Zkompilujte model s optimalizátorem **Adam**, ztrátovou funkcí **binary_crossentropy** a metrikou **accuracy**.
7. **Trénování sítě:** Zvolte vhodný počet epoch, velikost dávky (batch_size) a validační poměr (validation_split) a natrénujte síť.
8. **Predikce:** Proveďte odhady na testovací množině `X_test`.
9. **Vyhodnocení:** Vygenerujte souhrnný klasifikační report (`classification_report`) a matici záměn na testovací sadě.


In [ ]:
# Krok 1 & 2: Import knihoven a načtení dat
import os
os.environ['KERAS_BACKEND'] = 'torch'  # Podpora Keras 3

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import keras
from keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

# Načtení datasetu Sonar (soubor nemá hlavičku, má 60 číselných sloupců + 1 sloupec třídy)
try:
    sonar_df = pd.read_csv('sonar.csv', header=None)
except FileNotFoundError:
    try:
        sonar_df = pd.read_csv('data/sonar.csv', header=None)
    except FileNotFoundError:
        sonar_df = pd.read_csv('data/MAL_downloadable materials_session 2/Homework/sonar.csv', header=None)

print(f"Dataset úspěšně načten! Tvar: {sonar_df.shape} (208 vzorků, 61 sloupců)")
print(f"Zastoupení tříd v datech:\n{sonar_df[60].value_counts()}")
sonar_df.head(3)


### 🧹 Krok 3 & 4: Transformace proměnných & Train-Test Split
- **Cílová proměnná (Závislá):** Převod textových štítků `'M'` (Kovová mina) $\to$ 1 a `'R'` (Skála) $\to$ 0.
- **Vstupní signály (Nezávislé):** 60 frekvenčních pásem normalizujeme pomocí `StandardScaler` (nulový průměr, jednotkový rozptyl).
- **Rozdělení dat:** Rozdělení v poměru 80:20 se stratifikací a fixním `random_state=42`.


In [ ]:
# Krok 3 & 4: Příprava dat a Split
X_raw = sonar_df.iloc[:, :60].values
y_raw = sonar_df.iloc[:, 60].values

# Kódování cíle: 1 = Kovová mina (M), 0 = Skála (R)
y = np.where(y_raw == 'M', 1, 0)

# Rozdělení na trénovací a testovací sadu (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.20, random_state=42, stratify=y
)

# Normalizace příznaků pomocí StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Trénovací sada: {X_train_scaled.shape[0]} vzorků signálu")
print(f"Testovací sada : {X_test_scaled.shape[0]} vzorků signálu")


### 🧠 Krok 5 & 6: Návrh architektury sítě v Keras a kompilace
Pro poměr $p=60$ vstupů a pouze $n=166$ trénovacích vzorků je nutná rozumně kompaktní architektura s Dropoutem pro prevenci přeučení:
- **Vstupní vrstva:** 60 neuronů (frekvenční pásma).
- **Skrytá vrstva 1:** 32 neuronů, aktivace ReLU + Dropout (0.25).
- **Skrytá vrstva 2:** 16 neuronů, aktivace ReLU + Dropout (0.15).
- **Výstupní vrstva:** 1 neuron, aktivační funkce Sigmoid pro binární pravděpodobnost $P(\text{Mina})$.
- **Kompilace:** Optimalizátor **Adam** ($lr = 0.001$), ztrátová funkce **binary_crossentropy**, metrika **accuracy**.


In [ ]:
# Krok 5 & 6: Sestavení a kompilace Keras modelu
keras.utils.set_random_seed(42)

model = models.Sequential([
    layers.Input(shape=(60,)),
    layers.Dense(32, activation='relu', name="hidden_layer_1"),
    layers.Dropout(0.25, name="dropout_1"),
    layers.Dense(16, activation='relu', name="hidden_layer_2"),
    layers.Dropout(0.15, name="dropout_2"),
    layers.Dense(1, activation='sigmoid', name="output_layer")
], name="Sonar_MLP_Classifier")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


### ⚡ Krok 7: Trénování neuronové sítě
Trénujeme síť na normalizovaných trénovacích datech s parametry:
- `epochs = 60`
- `batch_size = 16`
- `validation_split = 0.15` (15 % trénovacích dat vyhrazeno pro validaci průběhu)


In [ ]:
# Krok 7: Trénování modelu
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.15,
    epochs=60,
    batch_size=16,
    verbose=1
)

# Vizualizace křivek učení
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss křivka
ax1.plot(history.history['loss'], label='Trénovací ztráta (Loss)', color='#2563eb', lw=2)
ax1.plot(history.history['val_loss'], label='Validační ztráta (Val Loss)', color='#ef4444', lw=2)
ax1.set_title('Vývoj ztrátové funkce (Binary Crossentropy)')
ax1.set_xlabel('Epocha')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy křivka
ax2.plot(history.history['accuracy'], label='Trénovací přesnost (Acc)', color='#10b981', lw=2)
ax2.plot(history.history['val_accuracy'], label='Validační přesnost (Val Acc)', color='#f59e0b', lw=2)
ax2.set_title('Vývoj přesnosti klasifikace (Accuracy)')
ax2.set_xlabel('Epocha')
ax2.set_ylabel('Přesnost')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 📋 Krok 8 & 9: Predikce na testovací sadě & Souhrnný Classification Report
Vygenerujeme pravděpodobnosti a binární predikce pro 42 neviděných testovacích měření sonaru.


In [ ]:
# Krok 8 & 9: Predikce a vyhodnocení na X_test
y_prob = model.predict(X_test_scaled).flatten()
y_pred = (y_prob >= 0.50).astype(int)

target_names = ['Skála (Rock - 0)', 'Mina (Mine - 1)']

print("=" * 60)
print("📊 TEST SET CLASSIFICATION REPORT (KERAS NEURAL NETWORK)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=target_names))
print("=" * 60)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
print(f"Celková přesnost (Accuracy): {acc*100:.2f} % ({int(acc*len(y_test))} ze {len(y_test)} správně)")
print(f"Plocha pod ROC křivkou (AUC): {auc:.4f}")

# Matice záměn
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Matice záměn na testovací sadě')
plt.xlabel('Predikce neuronové sítě')
plt.ylabel('Skutečný objekt')
plt.tight_layout()
plt.show()


### 🌊 Fyzikální interpretace: Akustické spektrální profily Skály vs. Miny
Proč dokáže neuronová síť rozeznat kovový válec od kamene?  
Zobrazíme průměrné odražené energie napříč všemi 60 frekvenčními úhly:
- **Kovová mina:** Vykazuje výrazné rezonanční píky v pásmech 25–45 díky zrcadlovému odrazu od hladkého válcového povrchu.
- **Přírodní skála:** Vykazuje difúzní rozptyl akustické vlny bez ostrých spektrálních špiček.


In [ ]:
# Porovnání průměrných frekvenčních spekter
rock_mean = sonar_df[sonar_df[60] == 'R'].iloc[:, :60].mean()
mine_mean = sonar_df[sonar_df[60] == 'M'].iloc[:, :60].mean()

plt.figure(figsize=(12, 5))
plt.plot(range(1, 61), rock_mean.values, label='Průměrný profil: Přírodní skála (R)', color='#6b7280', lw=2.5)
plt.plot(range(1, 61), mine_mean.values, label='Průměrný profil: Kovová mina (M)', color='#ef4444', lw=2.5)
plt.title('Akustický podpis: Spektrální odrazivost skály vs. kovové miny')
plt.xlabel('Frekvenční pásmo / Úhel sonaru (1 až 60)')
plt.ylabel('Normalizovaná odražená energie')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
